
# CHƯƠNG 6 — CODE CHO TẤT CẢ CÁC VÍ DỤ

Notebook này được xây dựng theo **Chương 6: Tính gần đúng đạo hàm và tích phân**.

Phạm vi:
- Giữ nguyên dữ liệu và công thức của các ví dụ đang hoạt động trong chương.
- Không đưa các ví dụ nằm trong môi trường `comment`.
- Không giải phần bài tập cuối chương.

Notebook gồm **16 ví dụ**:

1. Sai phân tiến cho \(f(x)=\ln x\).
2. Công thức 3-điểm và 5-điểm cho \(f(x)=xe^x\).
3. Vận tốc tức thời của xe tự hành.
4. Newton--Cotes đóng với \(n=4\).
5. Newton--Cotes đóng với \(n=5\) và đánh giá sai số.
6. Công thức hình thang và đánh giá sai số.
7. Công thức hình thang mở rộng với \(n=10\).
8. Tổng lượng nước từ dữ liệu bảng bằng hình thang mở rộng.
9. Hình thang mở rộng cho \(e^{x^2}\) và đánh giá sai số.
10. Simpson một phần ba và đánh giá sai số.
11. Simpson một phần ba mở rộng và đánh giá sai số.
12. Tích phân Romberg.
13. Gauss \(2\)-, \(3\)- và \(4\)-điểm cho \(e^x\).
14. Gauss \(4\)-điểm trên đoạn tổng quát.
15. Gauss \(3\)-điểm và đánh giá sai số.
16. Ví dụ tổng hợp so sánh hình thang, Simpson và Gauss.


In [1]:

import math
import numpy as np
import pandas as pd

from numpy.polynomial.legendre import leggauss

pd.set_option("display.precision", 12)
np.set_printoptions(precision=12, suppress=True)

# ------------------------------------------------------------
# Đạo hàm số
# ------------------------------------------------------------

def forward_difference(f, x0, h):
    return (f(x0 + h) - f(x0)) / h

def backward_difference(f, x0, h):
    return (f(x0) - f(x0 - h)) / h

def three_point_endpoint(f, x0, h):
    return (-3*f(x0) + 4*f(x0+h) - f(x0+2*h)) / (2*h)

def three_point_midpoint(f, x0, h):
    return (-f(x0-h) + f(x0+h)) / (2*h)

def five_point_endpoint(f, x0, h):
    return (
        -25*f(x0)
        + 48*f(x0+h)
        - 36*f(x0+2*h)
        + 16*f(x0+3*h)
        - 3*f(x0+4*h)
    ) / (12*h)

def five_point_midpoint(f, x0, h):
    return (
        f(x0-2*h)
        - 8*f(x0-h)
        + 8*f(x0+h)
        - f(x0+2*h)
    ) / (12*h)

# ------------------------------------------------------------
# Newton--Cotes và các công thức tích phân
# ------------------------------------------------------------

def cotes_weights(n):
    # H_i thỏa sum H_i i^k = n^(k+1)/(k+1), k=0,...,n
    nodes = np.arange(n + 1, dtype=float)
    A = np.vstack([nodes**k for k in range(n + 1)])
    rhs = np.array([n**(k+1)/(k+1) for k in range(n + 1)], dtype=float)
    return np.linalg.solve(A, rhs)

def closed_newton_cotes(f, a, b, n):
    h = (b - a) / n
    x = np.array([a + i*h for i in range(n + 1)], dtype=float)
    H = cotes_weights(n)
    fx = np.array([f(v) for v in x], dtype=float)
    return h*np.dot(H, fx), x, fx, H

def trapezoid(f, a, b):
    return (b-a)/2 * (f(a) + f(b))

def composite_trapezoid(f, a, b, n):
    h = (b-a)/n
    x = np.array([a + i*h for i in range(n+1)], dtype=float)
    fx = np.array([f(v) for v in x], dtype=float)
    I = h/2 * (fx[0] + 2*np.sum(fx[1:-1]) + fx[-1])
    return I, x, fx

def composite_trapezoid_from_table(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    h = np.diff(x)
    if not np.allclose(h, h[0]):
        raise ValueError("Các mốc phải cách đều.")

    step = h[0]
    I = step/2 * (y[0] + 2*np.sum(y[1:-1]) + y[-1])
    return I

def simpson_one_third(f, a, b):
    m = (a+b)/2
    return (b-a)/6 * (f(a) + 4*f(m) + f(b))

def composite_simpson_one_third(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("Số đoạn chia n phải là số chẵn.")

    h = (b-a)/n
    x = np.array([a + i*h for i in range(n+1)], dtype=float)
    fx = np.array([f(v) for v in x], dtype=float)

    odd_sum = np.sum(fx[1:-1:2])
    even_sum = np.sum(fx[2:-1:2])

    I = h/3 * (fx[0] + fx[-1] + 4*odd_sum + 2*even_sum)
    return I, x, fx, even_sum, odd_sum

def romberg(f, a, b, max_level):
    R = np.full((max_level+1, max_level+1), np.nan, dtype=float)
    R[0,0] = trapezoid(f, a, b)

    for i in range(1, max_level+1):
        n = 2**i
        h = (b-a)/n

        new_points_sum = sum(
            f(a + (2*j-1)*h)
            for j in range(1, 2**(i-1)+1)
        )

        R[i,0] = 0.5*R[i-1,0] + h*new_points_sum

        for k in range(1, i+1):
            R[i,k] = (
                4**k * R[i,k-1] - R[i-1,k-1]
            ) / (4**k - 1)

    return R

def gauss_legendre(f, a, b, n):
    nodes, weights = leggauss(n)
    c = (a+b)/2
    d = (b-a)/2
    x = c + d*nodes
    fx = np.array([f(v) for v in x], dtype=float)
    I = d*np.dot(weights, fx)
    return I, nodes, weights, x, fx

def gauss_error_bound(n, M):
    c = (
        2**(2*n+1) * math.factorial(n)**4
        /
        (
            math.factorial(2*n)**3
            * (2*n+1)
        )
    )
    return c*M



## Ví dụ 1 — Sai phân tiến cho \(f(x)=\ln x\)

Dùng công thức tỉ sai phân tiến để tính gần đúng đạo hàm của

\[
f(x)=\ln x
\]

tại

\[
x_0=1.8
\]

với

\[
h=0.1,\quad 0.05,\quad 0.01.
\]

Giá trị đúng:

\[
f'(1.8)=\frac1{1.8}.
\]

Vì

\[
f''(x)=-\frac1{x^2},
\]

nên trên đoạn \([1.8,1.8+h]\),

\[
M=\frac1{1.8^2}.
\]

Cận sai số:

\[
\Delta_D
\le
\frac{Mh}{2}.
\]


In [2]:

f = math.log
x0 = 1.8
hs = [0.1, 0.05, 0.01]

true_value = 1/x0
M = 1/x0**2

rows = []

for h in hs:
    approx = forward_difference(f, x0, h)
    true_error = abs(true_value - approx)
    bound = M*h/2

    rows.append([h, approx, true_error, bound])

df = pd.DataFrame(
    rows,
    columns=["h", "D(f;x0;h)", "Sai số thực", "Cận sai số Mh/2"]
)

print("f'(1.8) chính xác =", true_value)
df


f'(1.8) chính xác = 0.5555555555555556


,h,D(f;x0;h),Sai số thực,Cận sai số Mh/2
0,0.10,0.540672212703,0.014883342853,0.015432098765
1,0.05,0.547979483762,0.007576071793,0.007716049383
2,0.01,0.554018037562,0.001537517994,0.001543209877



## Ví dụ 2 — Công thức 3-điểm và 5-điểm

Giá trị của

\[
f(x)=xe^x
\]

được cho bởi bảng:

\[
\begin{array}{c|ccccc}
x&1.8&1.9&2.0&2.1&2.2\\
\hline
f(x)&10.889365&12.703199&14.778112&17.148957&19.855030
\end{array}
\]

Tính gần đúng \(f'(2)\) bằng:

- 3-điểm cuối với \(h=0.1\);
- 3-điểm cuối với \(h=-0.1\);
- 3-điểm giữa với \(h=0.1\);
- 3-điểm giữa với \(h=0.2\);
- 5-điểm giữa với \(h=0.1\).

Giá trị chính xác:

\[
f'(2)=3e^2.
\]


In [3]:

table = {
    1.8: 10.889365,
    1.9: 12.703199,
    2.0: 14.778112,
    2.1: 17.148957,
    2.2: 19.855030,
}

def f_tab(x):
    key = round(float(x), 10)
    if key not in table:
        raise KeyError(f"Không có dữ liệu tại x={x}")
    return table[key]

true_value = 3*math.e**2

methods = [
    ("3-điểm cuối, h=0.1", three_point_endpoint(f_tab, 2.0, 0.1)),
    ("3-điểm cuối, h=-0.1", three_point_endpoint(f_tab, 2.0, -0.1)),
    ("3-điểm giữa, h=0.1", three_point_midpoint(f_tab, 2.0, 0.1)),
    ("3-điểm giữa, h=0.2", three_point_midpoint(f_tab, 2.0, 0.2)),
    ("5-điểm giữa, h=0.1", five_point_midpoint(f_tab, 2.0, 0.1)),
]

rows = [
    [name, value, abs(value-true_value)]
    for name, value in methods
]

print("Giá trị chính xác f'(2) =", true_value)

pd.DataFrame(
    rows,
    columns=["Công thức", "Giá trị xấp xỉ", "Sai số tuyệt đối"]
)


Giá trị chính xác f'(2) = 22.16716829679195


,Công thức,Giá trị xấp xỉ,Sai số tuyệt đối
0,"3-điểm cuối, h=0.1",22.032310000000,0.134858296792
1,"3-điểm cuối, h=-0.1",22.054525000000,0.112643296792
2,"3-điểm giữa, h=0.1",22.228790000000,0.061621703208
3,"3-điểm giữa, h=0.2",22.414162500000,0.246994203208
4,"5-điểm giữa, h=0.1",22.166999166667,0.000169130125



## Ví dụ 3 — Vận tốc tức thời của xe tự hành

Bảng quãng đường:

\[
\begin{array}{c|ccccc}
t&8&9&10&11&12\\
\hline
s&73.60&86.85&100.80&115.55&131.20
\end{array}
\]

Dùng công thức 5-điểm giữa với

\[
h=1
\]

để tính

\[
v(10)=s'(10).
\]


In [4]:

s_table = {
    8.0: 73.60,
    9.0: 86.85,
    10.0: 100.80,
    11.0: 115.55,
    12.0: 131.20,
}

def s_tab(t):
    return s_table[round(float(t), 10)]

v10 = five_point_midpoint(s_tab, 10.0, 1.0)

print("v(10) ≈", v10, "m/s")


v(10) ≈ 14.333333333333337 m/s



## Ví dụ 4 — Newton--Cotes đóng với \(n=4\)

Tính gần đúng

\[
I=\int_0^1 \frac{1}{1+x^3}\,dx
\]

bằng công thức Newton--Cotes đóng với

\[
n=4.
\]

Theo bảng hệ số Cotes của chương,

\[
H=
\left(
\frac{14}{45},
\frac{64}{45},
\frac{8}{15},
\frac{64}{45},
\frac{14}{45}
\right).
\]


In [5]:

f = lambda x: 1/(1+x**3)

I_nc4, xvals, fvals, H = closed_newton_cotes(f, 0, 1, 4)

detail = pd.DataFrame({
    "i": np.arange(5),
    "x_i": xvals,
    "f(x_i)": fvals,
    "H_i": H,
    "H_i f(x_i)": H*fvals
})

display(detail)

print("Tổng H_i f(x_i) =", np.dot(H, fvals))
print("I_NC ≈", I_nc4)


,i,x_i,f(x_i),H_i,H_i f(x_i)
0,0,0.00,1.000000000000,0.311111111111,0.311111111111
1,1,0.25,0.984615384615,1.422222222222,1.400341880342
2,2,0.50,0.888888888889,0.533333333333,0.474074074074
3,3,0.75,0.703296703297,1.422222222222,1.000244200244
4,4,1.00,0.500000000000,0.311111111111,0.155555555556


Tổng H_i f(x_i) = 3.3413268213268212
I_NC ≈ 0.8353317053317053



## Ví dụ 5 — Newton--Cotes đóng với \(n=5\) và đánh giá sai số

Tính gần đúng

\[
I=
\int_{0.2}^{1.2}
\frac{x^3}{x+1}\,dx
\]

bằng Newton--Cotes đóng với

\[
n=5.
\]

Ở đây

\[
h=0.2.
\]

Theo chương,

\[
f^{(6)}(x)
=
-\frac{6!}{(x+1)^7},
\]

nên

\[
M
=
\frac{6!}{1.2^7}.
\]

Ngoài ra,

\[
\int_0^5
|t(t-1)\cdots(t-5)|\,dt
=
\frac{2459}{84}.
\]

Do đó,

\[
\Delta_{\rm NC}
\le
\frac{Mh^7}{6!}
\frac{2459}{84}.
\]


In [6]:

f = lambda x: x**3/(x+1)

I_nc5, xvals, fvals, H = closed_newton_cotes(f, 0.2, 1.2, 5)

M = math.factorial(6)/(1.2**7)
h = 0.2
abs_product_integral = 2459/84

error_bound = (
    M*h**7/math.factorial(6)
    * abs_product_integral
)

detail = pd.DataFrame({
    "i": np.arange(6),
    "x_i": xvals,
    "f(x_i)": fvals,
    "H_i": H,
    "H_i f(x_i)": H*fvals
})

display(detail)

print("I_NC ≈", I_nc5)
print("M =", M)
print("Cận sai số ≈", error_bound)


,i,x_i,f(x_i),H_i,H_i f(x_i)
0,0,0.2,0.006666666667,0.329861111111,0.002199074074
1,1,0.4,0.045714285714,1.302083333333,0.059523809524
2,2,0.6,0.135000000000,0.868055555555,0.117187500000
3,3,0.8,0.284444444444,0.868055555556,0.246913580247
4,4,1.0,0.500000000000,1.302083333333,0.651041666667
5,5,1.2,0.785454545455,0.329861111111,0.259090909091


I_NC ≈ 0.2671913079204746
M = 200.9387860082305
Cận sai số ≈ 0.00010457322217867492



## Ví dụ 6 — Công thức hình thang

Tính gần đúng

\[
I=
\int_0^1
\frac{x+2}{x+1}\,dx
\]

bằng công thức hình thang.

Với

\[
f''(x)
=
\frac{2}{(x+1)^3},
\]

ta có

\[
M=2.
\]

Sai số được đánh giá bởi

\[
\Delta_{\rm HT}
\le
\frac{M(b-a)^3}{12}.
\]


In [7]:

f = lambda x: (x+2)/(x+1)

I_ht = trapezoid(f, 0, 1)

M = 2
error_bound = M*(1-0)**3/12

print("I_HT ≈", I_ht)
print("Cận sai số =", error_bound)


I_HT ≈ 1.75
Cận sai số = 0.16666666666666666



## Ví dụ 7 — Hình thang mở rộng với \(n=10\)

Tính gần đúng

\[
I=
\int_{0.2}^{1.2}
\frac{x^2+1}{\sin^2x+1}\,dx
\]

bằng công thức hình thang mở rộng với

\[
n=10.
\]

Bước chia:

\[
h=0.1.
\]


In [8]:

f = lambda x: (x**2 + 1)/(math.sin(x)**2 + 1)

I_htmr, xvals, fvals = composite_trapezoid(f, 0.2, 1.2, 10)

detail = pd.DataFrame({
    "i": np.arange(11),
    "x_i": xvals,
    "f(x_i)": fvals
})

display(detail)

print("I_HTMR ≈", I_htmr)


,i,x_i,f(x_i)
0,0,0.2,1.000510353599
1,1,0.3,1.002453534875
2,2,0.4,1.007253400778
3,3,0.5,1.016385064703
4,4,0.6,1.031224004930
5,5,0.7,1.052991308042
6,6,0.8,1.082794307820
7,7,0.9,1.121714690863
8,8,1.0,1.170909855866
9,9,1.1,1.231712031173


I_HTMR ≈ 1.087055475083737



## Ví dụ 8 — Tổng lượng nước từ dữ liệu bảng

Lưu lượng nước trong 30 phút:

\[
\begin{array}{c|ccccccc}
t\;(\text{phút})&0&5&10&15&20&25&30\\
\hline
Q(t)\;(\text{lít/phút})
&18.4&20.1&21.5&22.0&21.2&19.8&18.9
\end{array}
\]

Tổng lượng nước:

\[
V=
\int_0^{30}Q(t)\,dt.
\]

Dùng công thức hình thang mở rộng với

\[
h=5.
\]


In [9]:

t = np.array([0, 5, 10, 15, 20, 25, 30], dtype=float)
Q = np.array([18.4, 20.1, 21.5, 22.0, 21.2, 19.8, 18.9], dtype=float)

V = composite_trapezoid_from_table(t, Q)

print("Tổng lượng nước V ≈", V, "lít")


Tổng lượng nước V ≈ 616.25 lít



## Ví dụ 9 — Hình thang mở rộng cho \(e^{x^2}\)

Tính gần đúng

\[
I=
\int_1^2 e^{x^2}\,dx
\]

bằng công thức hình thang mở rộng với

\[
n=8,
\qquad
h=0.125.
\]

Ta có

\[
f''(x)=2e^{x^2}(1+2x^2).
\]

Trên \([1,2]\),

\[
M=18e^4.
\]

Cận sai số:

\[
\Delta_{\rm HTMR}
\le
\frac{(b-a)Mh^2}{12}.
\]


In [10]:

f = lambda x: math.exp(x**2)

I_htmr, xvals, fvals = composite_trapezoid(f, 1, 2, 8)

h = 1/8
M = 18*math.e**4
error_bound = (2-1)*M*h**2/12

detail = pd.DataFrame({
    "i": np.arange(9),
    "x_i": xvals,
    "f(x_i)": fvals
})

display(detail)

print("I_HTMR ≈", I_htmr)
print("M =", M)
print("Cận sai số ≈", error_bound)


,i,x_i,f(x_i)
0,0,1.000,2.718281828459
1,1,1.125,3.545307861224
2,2,1.250,4.770733181968
3,3,1.375,6.623507079584
4,4,1.500,9.487735836359
5,5,1.625,14.021964597513
6,6,1.750,21.380942759123
7,7,1.875,33.636944445854
8,8,2.000,54.598150033144


I_HTMR ≈ 15.265668961553121
M = 982.7667005965961
Cận sai số ≈ 1.2796441414018178



## Ví dụ 10 — Simpson một phần ba

Tính gần đúng

\[
I=
\int_{0.6}^{1}
\frac{x^2}{x+1}\,dx
\]

bằng công thức Simpson một phần ba.

Theo chương,

\[
M
=
\max_{0.6\le x\le1}
\frac{4!}{(x+1)^5}
=
\frac{24}{1.6^5}.
\]

Cận sai số:

\[
\Delta_{\rm S1/3}
\le
\frac{M(b-a)^5}{2880}.
\]


In [11]:

f = lambda x: x**2/(x+1)

I_s13 = simpson_one_third(f, 0.6, 1.0)

M = math.factorial(4)/(1.6**5)
error_bound = M*(1.0-0.6)**5/2880

print("I_S1/3 ≈", I_s13)
print("M =", M)
print("Cận sai số ≈", error_bound)


I_S1/3 ≈ 0.14314814814814816
M = 2.2888183593749996
Cận sai số ≈ 8.138020833333333e-06



## Ví dụ 11 — Simpson một phần ba mở rộng

Tính gần đúng

\[
I=
\int_1^2
\ln(2x+1)\,dx
\]

bằng Simpson một phần ba mở rộng với

\[
n=10,
\qquad
h=0.1.
\]

Ta có

\[
|f^{(4)}(x)|
=
\frac{2^4 3!}{(2x+1)^4}.
\]

Do đó trên \([1,2]\),

\[
M=
\frac{96}{3^4}.
\]

Sai số:

\[
\Delta_{\rm S1/3MR}
\le
\frac{(b-a)Mh^4}{180}.
\]


In [12]:

f = lambda x: math.log(2*x + 1)

I_smr, xvals, fvals, sigma0, sigma1 = composite_simpson_one_third(
    f, 1, 2, 10
)

h = 0.1
M = 96/(3**4)
error_bound = (2-1)*M*h**4/180

detail = pd.DataFrame({
    "i": np.arange(11),
    "x_i": xvals,
    "f(x_i)": fvals
})

display(detail)

print("sigma_0 =", sigma0)
print("sigma_1 =", sigma1)
print("I_S1/3MR ≈", I_smr)
print("M =", M)
print("Cận sai số ≈", error_bound)


,i,x_i,f(x_i)
0,0,1.0,1.098612288668
1,1,1.1,1.163150809806
2,2,1.2,1.223775431622
3,3,1.3,1.280933845462
4,4,1.4,1.335001066732
5,5,1.5,1.386294361120
6,6,1.6,1.435084525289
7,7,1.7,1.481604540924
8,8,1.8,1.526056303495
9,9,1.9,1.568615917914


sigma_0 = 5.5199173271388275
sigma_1 = 6.880599475225697
I_S1/3MR ≈ 1.3756760918760884
M = 1.1851851851851851
Cận sai số ≈ 6.584362139917696e-07



## Ví dụ 12 — Phương pháp tích phân Romberg

Tính gần đúng

\[
I=
\int_0^1
\frac{1}{1+x^2}\,dx
\]

bằng Romberg với số đoạn chia tối đa

\[
n=4.
\]

Giá trị chính xác:

\[
I=\frac{\pi}{4}.
\]

Ta cần các mức hình thang tương ứng với

\[
n=1,\quad2,\quad4,
\]

tức mức Romberg tối đa \(m=2\).


In [13]:

f = lambda x: 1/(1+x**2)

R = romberg(f, 0, 1, max_level=2)
exact = math.pi/4

romberg_table = pd.DataFrame({
    "n": [1, 2, 4],
    "h": [1, 0.5, 0.25],
    "R_i0": [R[0,0], R[1,0], R[2,0]],
    "R_i1": [np.nan, R[1,1], R[2,1]],
    "R_i2": [np.nan, np.nan, R[2,2]],
})

display(romberg_table)

print("Giá trị chính xác =", exact)
print("Sai số hình thang n=4 =", abs(exact - R[2,0]))
print("Sai số Romberg cuối =", abs(exact - R[2,2]))


,n,h,R_i0,R_i1,R_i2
0,1,1.00,0.750000000000,NaN,NaN
1,2,0.50,0.775000000000,0.783333333333,NaN
2,4,0.25,0.782794117647,0.785392156863,0.785529411765


Giá trị chính xác = 0.7853981633974483
Sai số hình thang n=4 = 0.0026040457503895276
Sai số Romberg cuối = 0.00013124836725753042



## Ví dụ 13 — Gauss \(2\)-, \(3\)- và \(4\)-điểm cho \(e^x\)

Tính gần đúng

\[
I=
\int_{-1}^{1}e^x\,dx.
\]

Giá trị chính xác:

\[
I=e-e^{-1}.
\]

So sánh các công thức Gauss \(2\)-, \(3\)- và \(4\)-điểm.


In [14]:

f = math.exp
exact = math.e - math.e**(-1)

rows = []

for n in [2, 3, 4]:
    approx, nodes, weights, x_nodes, fx = gauss_legendre(f, -1, 1, n)
    rows.append([
        n,
        approx,
        abs(exact-approx)
    ])

print("Giá trị chính xác =", exact)

pd.DataFrame(
    rows,
    columns=["Số điểm Gauss", "Giá trị xấp xỉ", "Sai số tuyệt đối"]
)


Giá trị chính xác = 2.3504023872876028


,Số điểm Gauss,Giá trị xấp xỉ,Sai số tuyệt đối
0,2,2.342696087910,0.007706299378
1,3,2.350336928680,0.000065458608
2,4,2.350402092156,0.000000295131



## Ví dụ 14 — Gauss \(4\)-điểm trên đoạn tổng quát

Tính gần đúng

\[
I=
\int_1^{1.5}x\ln x\,dx
\]

bằng cầu phương Gauss \(4\)-điểm.

Đổi biến:

\[
x=
0.25t+1.25.
\]

Do đó

\[
I
=
0.25
\int_{-1}^{1}
(0.25t+1.25)
\ln(0.25t+1.25)\,dt.
\]


In [15]:

f = lambda x: x*math.log(x)

I_g4, nodes, weights, x_phys, fx = gauss_legendre(f, 1, 1.5, 4)

detail = pd.DataFrame({
    "t_i": nodes,
    "B_i": weights,
    "x_i trên [1,1.5]": x_phys,
    "f(x_i)": fx
})

display(detail)

print("I_G4 ≈", I_g4)


,t_i,B_i,"x_i trên [1,1.5]",f(x_i)
0,-0.861136311594,0.347854845137,1.034715922101,0.035311665058
1,-0.339981043585,0.652145154863,1.165004739104,0.177925529250
2,0.339981043585,0.652145154863,1.334995260896,0.385717166252
3,0.861136311594,0.347854845137,1.465284077899,0.559810512291


I_G4 ≈ 0.1436482464462769



## Ví dụ 15 — Gauss \(3\)-điểm và đánh giá sai số

Tính gần đúng

\[
I=
\int_{-1}^{1}
\frac{x+2}{x+3}\,dx
\]

bằng Gauss \(3\)-điểm.

Theo chương,

\[
f^{(6)}(x)
=
-\frac{6!}{(x+3)^7}
\]

về dấu, nên

\[
M
=
\max_{-1\le x\le1}
|f^{(6)}(x)|
=
\frac{6!}{2^7}
=
\frac{45}{8}.
\]

Cận sai số:

\[
\Delta_G
\le
\frac{
2^7(3!)^4
}{
(6!)^3\,7
}
M.
\]


In [16]:

f = lambda x: (x+2)/(x+3)

I_g3, nodes, weights, x_phys, fx = gauss_legendre(f, -1, 1, 3)

M = math.factorial(6)/(2**7)
error_bound = gauss_error_bound(3, M)

print("I_G3 ≈", I_g3)
print("M =", M)
print("Cận sai số ≈", error_bound)


I_G3 ≈ 1.306878306878307
M = 5.625
Cận sai số ≈ 0.00035714285714285714



## Ví dụ 16 — So sánh hình thang, Simpson và Gauss

Lưu lượng:

\[
Q(t)
=
20
+
5\sin\left(\frac{\pi t}{10}\right),
\qquad
0\le t\le10.
\]

Tổng lượng nước:

\[
V=
\int_0^{10}Q(t)\,dt.
\]

Giá trị chính xác:

\[
V
=
200+\frac{100}{\pi}.
\]

So sánh:

1. Hình thang mở rộng với \(n=10\);
2. Simpson một phần ba mở rộng với \(n=10\);
3. Gauss \(2\)-điểm;
4. Gauss \(3\)-điểm;
5. Gauss \(4\)-điểm.


In [17]:

Q = lambda t: 20 + 5*math.sin(math.pi*t/10)

exact = 200 + 100/math.pi

I_trap, _, _ = composite_trapezoid(Q, 0, 10, 10)
I_simp, _, _, _, _ = composite_simpson_one_third(Q, 0, 10, 10)

I_g2, *_ = gauss_legendre(Q, 0, 10, 2)
I_g3, *_ = gauss_legendre(Q, 0, 10, 3)
I_g4, *_ = gauss_legendre(Q, 0, 10, 4)

results = [
    ("Hình thang mở rộng, n=10", I_trap),
    ("Simpson 1/3 mở rộng, n=10", I_simp),
    ("Gauss 2-điểm", I_g2),
    ("Gauss 3-điểm", I_g3),
    ("Gauss 4-điểm", I_g4),
]

rows = [
    [name, value, abs(exact-value)]
    for name, value in results
]

print("Giá trị chính xác =", exact)

pd.DataFrame(
    rows,
    columns=["Phương pháp", "Giá trị xấp xỉ", "Sai số tuyệt đối"]
)


Giá trị chính xác = 231.83098861837908


,Phương pháp,Giá trị xấp xỉ,Sai số tuyệt đối
0,"Hình thang mở rộng, n=10",231.568757573375,0.262231045004
1,"Simpson 1/3 mở rộng, n=10",231.832731640583,0.001743022204
2,Gauss 2-điểm,230.809525423978,1.021463194401
3,Gauss 3-điểm,231.853093864999,0.022105246620
4,Gauss 4-điểm,231.830737606488,0.000251011891



# Ghi chú

- Notebook bám theo các công thức và dữ liệu trong Chương 6.
- Với các ví dụ dùng dữ liệu bảng, code sử dụng đúng các giá trị đã cho trong bảng.
- Hàm `cotes_weights` xác định hệ số Newton--Cotes đóng từ hệ moment, nên không cần nhập tay bảng hệ số.
- Hàm `leggauss` của NumPy được dùng để lấy các nút và trọng số Gauss--Legendre với độ chính xác máy.
- Phần bài tập cuối chương và các ví dụ nằm trong môi trường `comment` không được đưa vào notebook.
